# Legacy EEG 模型：现场快速测试

这个 Notebook 只需要一个 EDF 路径，就会复用仓库当前旧模型的正式推理链：

```text
EDF → MNE 读取/EEG 通道选择/必要时重采样
    → 0.5–43 Hz FIR → 4 s 窗口、2 s 步长
    → Welch 频带特征 → 冻结的 StandardScaler → PCA → RBF SVC
    → 窗口预测、类别分布、decision score 和可用时的效果指标
```

重要边界：本 Notebook 只做 inference，不训练、不 fit、不微调、不做个体校准，也不写入正式 artifacts。文件名真值只用于评估，不参与模型输入。

## 使用方法

1. 执行导入和配置单元格。
2. 在 `EDF_PATH` 中填写路径；也可以在本地桌面环境执行 `choose_edf_file()` 后选择 EDF。
3. 执行最后一个单元格，查看窗口级预测和汇总。

默认路径是仓库中一个已存在、符合 `被试_状态_时间戳.edf` 字段顺序的示例：`lyc_focus_202609072034_raw.edf`。它可以直接替换成现场文件。`_raw` 等额外后缀不会改变第二个下划线字段的解析。

In [23]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# Notebook 可能从仓库根目录或 notebooks/ 目录启动。向上查找正式脚本，
# 这样不依赖用户当前工作目录，也不复制一套 EEG 处理代码。
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "scripts" / "eeg_pipeline_utils.py").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("无法定位 EEGAttention 仓库根目录")
    REPO_ROOT = REPO_ROOT.parent

SCRIPTS_DIR = REPO_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# 这两个 import 是复用正式实现的关键：
# eeg_utils 提供 MNE/预处理/窗口/Welch 特征；baseline 提供冻结协议常量。
import eeg_pipeline_utils as eeg_utils
import legacy_baseline_v0 as baseline

ARTIFACT_DIR = REPO_ROOT / "artifacts" / baseline.BASELINE_VERSION
PIPELINE_PATH = ARTIFACT_DIR / "pipeline.joblib"
CONFIG_PATH = ARTIFACT_DIR / "config.json"
FREEZE_MANIFEST_PATH = ARTIFACT_DIR / "freeze_manifest.json"

print(f"仓库根目录: {REPO_ROOT}")
print(f"冻结模型: {PIPELINE_PATH}")

仓库根目录: C:\CHLight\0-Plan\EEGreproduction\EEGAttention
冻结模型: C:\CHLight\0-Plan\EEGreproduction\EEGAttention\artifacts\legacy_baseline_v0\pipeline.joblib


## 1. 只填写或选择一个 EDF 路径

输入文件名优先遵循 `subject_status_timestamp.edf`。下面的选择器是可选便利功能；核心输入始终是一个路径字符串，不需要 CSV 或额外元数据文件。

In [24]:
# 现场使用时把这个字符串替换成 EDF 路径。相对路径相对于仓库根目录解析。
EDF_PATH = r"C:\Users\13313\Desktop\lyc_unfocus_202609072312.edf"
def choose_edf_file(initial_dir: Path | None = None) -> str:
    """在本地桌面 Jupyter 中弹出文件选择器；无 GUI 时给出可读提示。"""
    try:
        from tkinter import Tk, filedialog

        root = Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            initialdir=str(initial_dir or REPO_ROOT / "data"),
            title="选择一个 EDF 文件",
            filetypes=[("EDF files", "*.edf"), ("All files", "*.*")],
        )
        root.destroy()
        return selected
    except Exception as exc:
        print(f"当前环境无法打开图形文件选择器：{exc}")
        print("请直接在 EDF_PATH 中填写路径。")
        return ""

# 如果希望选择文件，可以取消下一行注释：
# EDF_PATH = choose_edf_file()
print(f"待测 EDF: {EDF_PATH or '尚未填写'}")

待测 EDF: C:\Users\13313\Desktop\lyc_unfocus_202609072312.edf


## 2. 严格解析 EDF 文件名中的标签

不使用模糊的 `"focus" in filename` 判断，而是读取 `Path(path).stem`，再用 `_` 分割：

```text
lyc_focus_202609072034_raw
│   │      │              │
│   │      │              └─ 额外后缀，可存在
│   │      └─ timestamp
│   └─ status 字段
└─ subject 字段
```

`focus` 可直接作为模型任务真值；`iu`、`ou` 按正式 baseline 的 `LABEL_MAPPING` 合并到 `unfocus`；`daze` 是仓库中出现过的原始状态，但当前冻结模型是 focus/unfocus 二分类，因此识别到 `daze` 时会警告并跳过准确率/F1/confusion matrix。

In [25]:
# 集中维护原始文件名状态到当前冻结二分类标签的映射。
# baseline.LABEL_MAPPING 是正式训练脚本当前使用的映射；这里仅补充
# 文件名解析需要知道的 unfocus 和 daze，不复制任何信号处理逻辑。
RAW_STATUS_TO_MODEL_LABEL = {
    **baseline.LABEL_MAPPING,
    "unfocus": "unfocus",
    "daze": None,  # 可识别，但不是当前二分类模型的可比真值
}
ALLOWED_RAW_STATUS = frozenset(RAW_STATUS_TO_MODEL_LABEL)
MODEL_LABELS = tuple(baseline.LABELS)

def parse_edf_filename(edf_path: Path) -> dict[str, object]:
    """按 stem 的 subject/status/timestamp 字段解析文件名，不做模糊匹配。"""
    stem = edf_path.stem
    fields = stem.split("_")
    parsed: dict[str, object] = {
        "filename": edf_path.name,
        "subject": fields[0] if fields else "",
        "raw_status": fields[1].casefold() if len(fields) >= 2 else "",
        "timestamp": fields[2] if len(fields) >= 3 else "",
        "true_label": None,
        "label_warning": None,
    }

    # 至少需要 subject、status、timestamp 三段；timestamp 也要求为数字，
    # 这样不会把任意后缀误当成时间戳。
    if len(fields) < 3 or not parsed["subject"] or not str(parsed["timestamp"]).isdigit():
        parsed["label_warning"] = (
            "文件名不符合 subject_status_timestamp 结构，"
            "无法得到可靠真值；仍会继续预测。"
        )
        print(f"⚠️ {parsed['label_warning']} 文件名: {edf_path.name}")
        return parsed

    raw_status = str(parsed["raw_status"])
    if raw_status not in ALLOWED_RAW_STATUS:
        parsed["label_warning"] = (
            f"status={raw_status!r} 不在允许集合 {sorted(ALLOWED_RAW_STATUS)} 中，"
            "无法得到可靠真值；仍会继续预测。"
        )
    else:
        mapped_label = RAW_STATUS_TO_MODEL_LABEL[raw_status]
        if mapped_label in MODEL_LABELS:
            parsed["true_label"] = mapped_label
        else:
            parsed["label_warning"] = (
                f"status={raw_status!r} 已识别，但当前冻结模型只支持 {MODEL_LABELS} 二分类；"
                "不计算准确率/F1。"
            )

    if parsed["label_warning"]:
        print(f"⚠️ {parsed['label_warning']} 文件名: {edf_path.name}")
    else:
        print(
            f"文件名真值: subject={parsed['subject']}, "
            f"raw_status={parsed['raw_status']}, true_label={parsed['true_label']}, "
            f"timestamp={parsed['timestamp']}"
        )
    return parsed

## 3. 加载冻结的旧模型

模型文件已经包含训练后的 StandardScaler、PCA 和 SVC 状态。这里做完整性与结构检查，但绝不调用 `fit()`；Notebook 后续只会调用 `predict()` 和 `decision_function()`。

In [26]:
def load_frozen_pipeline():
    """读取并验证 legacy_baseline_v0 的冻结 Pipeline。"""
    for path in (PIPELINE_PATH, CONFIG_PATH, FREEZE_MANIFEST_PATH):
        if not path.exists():
            raise FileNotFoundError(f"缺少冻结模型文件: {path}")

    config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    freeze_manifest = json.loads(FREEZE_MANIFEST_PATH.read_text(encoding="utf-8"))
    for name, path in {"pipeline.joblib": PIPELINE_PATH, "config.json": CONFIG_PATH}.items():
        expected = freeze_manifest.get("artifact_sha256", {}).get(name)
        if expected and eeg_utils.sha256_file(path) != expected:
            raise AssertionError(f"冻结文件 SHA-256 不匹配: {name}")

    pipeline = joblib.load(PIPELINE_PATH)
    if list(pipeline.named_steps) != ["scaler", "pca", "svc"]:
        raise AssertionError(f"非预期的 Pipeline 步骤: {list(pipeline.named_steps)}")
    if not all(hasattr(pipeline.named_steps[name], attr) for name, attr in (("scaler", "mean_"), ("pca", "components_"), ("svc", "support_"))):
        raise AssertionError("冻结 Pipeline 看起来尚未 fit")
    if int(pipeline.named_steps["scaler"].n_features_in_) != 240:
        raise AssertionError("冻结模型不是当前 240 维特征协议")
    if config.get("labels") != list(MODEL_LABELS):
        raise AssertionError("config.json 的标签顺序与正式 baseline 不一致")

    print(f"已加载冻结模型: {PIPELINE_PATH.name}")
    print(f"Pipeline: {list(pipeline.named_steps)}; features: {pipeline.named_steps['scaler'].n_features_in_}")
    return pipeline, config

PIPELINE, MODEL_CONFIG = load_frozen_pipeline()

已加载冻结模型: pipeline.joblib
Pipeline: ['scaler', 'pca', 'svc']; features: 240


c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\CHLight\1-Workconfig\Miniconda\envs\EEG\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.9.1 when using version 1.

## 4. 读取 EDF、复用正式特征链并输出结果

单个现场 EDF 没有 manifest 的 `activity_start_s`/`activity_end_s` 时，这里把整个 EDF 作为一个逻辑 segment（`0` 到 recording duration）。预处理和特征计算本身仍全部调用正式工具函数；因此不存在 Notebook 自己实现的第二套滤波、窗口或频带逻辑。

未知或不兼容标签时，函数会完成读取、特征、预测、类别分布和 decision score，但 `metrics` 会保持 `None`，不会输出或宣称准确率。

In [27]:
def run_quick_test(edf_path: str | Path) -> dict[str, object]:
    """对一个 EDF 完成旧模型 inference，并按真值可用性输出结果。"""
    path = Path(edf_path).expanduser()
    if not path.is_absolute():
        path = REPO_ROOT / path
    path = path.resolve()
    if not path.exists():
        raise FileNotFoundError(f"EDF 不存在: {path}")
    if path.suffix.casefold() != ".edf":
        raise ValueError(f"输入必须是 .edf 文件: {path}")

    filename_info = parse_edf_filename(path)

    # allow_locked=True 只是允许这个独立推理 Notebook 读取 data/locked；
    # 它不会改变预处理，也不会开启训练。正式 baseline 的默认保护仍在。
    data, sfreq, channels = eeg_utils.load_eeg_recording(path, allow_locked=True)
    duration_sec = data.shape[1] / sfreq
    X, window_starts = eeg_utils.extract_segment_features(
        data,
        sfreq,
        0.0,
        duration_sec,
        baseline.BANDS,
        window_sec=baseline.WINDOW_SEC,
        step_sec=baseline.STEP_SEC,
        l_freq=baseline.FILTER_L_HZ,
        h_freq=baseline.FILTER_H_HZ,
    )
    expected_features = int(PIPELINE.named_steps["scaler"].n_features_in_)
    if X.ndim != 2 or X.shape[1] != expected_features:
        raise AssertionError(f"特征形状 {X.shape} 与冻结模型维度 {expected_features} 不一致")

    # 旧模型已经 fit 好；此处只做推理。
    predicted = np.asarray(PIPELINE.predict(X))
    decision = np.asarray(PIPELINE.decision_function(X))
    if decision.ndim == 1:
        decision_score = decision.astype(float)
    elif decision.ndim == 2 and decision.shape[1] == 2:
        decision_score = decision[:, 1].astype(float)
    else:
        decision_score = np.full(len(predicted), np.nan, dtype=float)
        print(f"⚠️ 无法把 decision_function 解释成一维二分类 score，原始形状: {decision.shape}")

    predictions = pd.DataFrame(
        {
            "window_start_s": window_starts,
            "window_end_s": window_starts + baseline.WINDOW_SEC,
            "predicted_label": predicted,
            "decision_score": decision_score,
        }
    )
    counts = predictions["predicted_label"].value_counts().reindex(MODEL_LABELS, fill_value=0)
    distribution = pd.DataFrame(
        {"predicted_windows": counts.astype(int), "proportion": (counts / len(predictions)).astype(float)}
    )

    print(f"\nEDF: {path.name}")
    print(f"EEG channels: {len(channels)}; shape: {data.shape}; sampling rate: {sfreq:g} Hz")
    print(f"duration: {duration_sec:.2f} s; feature matrix: {X.shape}")
    print(f"window total: {len(predictions):,}")
    print("\nPrediction distribution:")
    display(distribution)
    print("Decision score summary:")
    display(predictions["decision_score"].describe()[["min", "25%", "50%", "75%", "max"]].to_frame().T)

    true_label = filename_info.get("true_label")
    metrics: dict[str, object] | None = None
    if true_label in MODEL_LABELS:
        y_true = np.full(len(predicted), str(true_label), dtype=object)
        cm = confusion_matrix(y_true, predicted, labels=list(MODEL_LABELS))
        report = classification_report(
            y_true,
            predicted,
            labels=list(MODEL_LABELS),
            target_names=list(MODEL_LABELS),
            output_dict=True,
            zero_division=0,
        )
        metrics = {
            "window_total": int(len(predicted)),
            "accuracy": float(accuracy_score(y_true, predicted)),
            "f1_focus": float(f1_score(y_true, predicted, average="binary", pos_label="focus", zero_division=0)),
            "f1_macro": float(f1_score(y_true, predicted, labels=list(MODEL_LABELS), average="macro", zero_division=0)),
            "confusion_matrix_label_order": list(MODEL_LABELS),
            "confusion_matrix": cm.astype(int).tolist(),
            "classification_report": report,
        }
        print(f"\nFilename truth: {true_label}")
        print(f"Accuracy: {metrics['accuracy']:.2%}")
        print(f"F1 (focus): {metrics['f1_focus']:.4f}")
        print(f"F1 (macro): {metrics['f1_macro']:.4f}")
        print("Confusion matrix [rows=true, columns=predicted; unfocus, focus]:")
        display(pd.DataFrame(cm, index=MODEL_LABELS, columns=MODEL_LABELS))
    else:
        print("\n⚠️ 没有可用于当前二分类模型的可靠真值；只展示预测分布和 decision score，不计算准确率/F1/confusion matrix。")

    return {
        "path": str(path),
        "filename_info": filename_info,
        "channels": channels,
        "sampling_rate_hz": float(sfreq),
        "features": X,
        "predictions": predictions,
        "distribution": distribution,
        "metrics": metrics,
    }

## 5. 执行一次现场测试

最后一个单元格是唯一需要反复执行的地方：换 `EDF_PATH` 后重新执行即可。函数只返回内存中的结果，不把 CSV、JSON 或模型写回仓库。

In [28]:
if not EDF_PATH:
    print("请先在上方填写 EDF_PATH，或取消 choose_edf_file() 注释选择文件。")
else:
    RESULT = run_quick_test(EDF_PATH)
    print("\n已完成：RESULT['predictions'] 保存窗口级预测，RESULT['metrics'] 保存可用时的指标。")

文件名真值: subject=lyc, raw_status=unfocus, true_label=unfocus, timestamp=202609072312

EDF: lyc_unfocus_202609072312.edf
EEG channels: 24; shape: (24, 116544); sampling rate: 128 Hz
duration: 910.50 s; feature matrix: (454, 240)
window total: 454

Prediction distribution:


,predicted_windows,proportion
predicted_label,,
unfocus,270,0.594714
focus,184,0.405286


Decision score summary:


,min,25%,50%,75%,max
decision_score,-1.5365,-0.308855,0.180168,0.606044,2.088739



Filename truth: unfocus
Accuracy: 59.47%
F1 (focus): 0.0000
F1 (macro): 0.3729
Confusion matrix [rows=true, columns=predicted; unfocus, focus]:


,unfocus,focus
unfocus,270,184
focus,0,0



已完成：RESULT['predictions'] 保存窗口级预测，RESULT['metrics'] 保存可用时的指标。
